# GameTheory-22 — Jeux stochastiques : la valeur d’un jeu à états (Shapley, 1953)

**Navigation** : [Index](README.md) | [<< Carnet 17e — persuasion bayésienne](GameTheory-17e-Bayesian-Persuasion-Python.ipynb) | [Version Python](GameTheory-22-Jeux-Stochastiques-Shapley-Python.ipynb)

Le carnet **Dynamics-01** rejoue une même matrice à chaque manche. Ici, l’état du jeu peut changer : après avoir choisi leurs actions, les joueurs reçoivent le gain courant puis passent à un nouvel état selon une transition qui peut dépendre de leurs actions.

### Objectifs d’apprentissage

- Décrire un jeu stochastique à somme nulle par ses états, gains et transitions.
- Résoudre les jeux matriciels 2×2 en distinguant point selle pur et équilibre mixte.
- Calculer la valeur dynamique par itération de Bellman, mesurer son écart à la valeur statique, et distinguer cet écart d’un gain de planification comme d’une valeur de l’information.
- Examiner ce qui change lorsque l’adversaire peut contester une transition.

### Prérequis

Jeux à somme nulle et minimax (carnet 05), jeux répétés (Dynamics-01), tableaux et méthodes C#.

### Durée estimée : 60 minutes

Ce jumeau recode les calculs du carnet Python en C# et BCL .NET, sans lancer Python ni appeler un solveur externe.

## 1. Modèle du jeu stochastique

Un jeu à somme nulle est décrit ici par $(S,A,B,G,P)$ : un ensemble fini d’états, les actions du rang (qui maximise) et de la colonne (qui minimise), une matrice de gains $G[s]$ par état, et des probabilités de transition $P[s,a,b,s']$.

Avec $0\leq\gamma<1$, l’opérateur de Bellman évalue chaque paire d’actions par

$$M_s[a,b]=(1-\gamma)G[s,a,b]+\gamma\sum_{s'}P[s,a,b,s']V(s')$$

puis prend la valeur minimax du jeu matriciel $M_s$, y compris les stratégies mixtes. Le théorème de Shapley garantit un point fixe unique dans le jeu fini actualisé; l’itération de valeur est une méthode numérique pour l’approcher.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Security;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;

// Separateur decimal invariant (CultureInfo) -> sorties reproductibles quelle que soit la machine.
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

record ValueIterationResult(double[] Values, int Iterations, List<double[]> History, bool Converged, double Residual);

Console.WriteLine($"Runtime .NET : {Environment.Version}; noyau : .NET Interactive / C#; framework cible du dépôt : .NET 9+");

// Gains G[état, action du rang, action de la colonne].
double[,,] Gains = new double[2, 2, 2]
{
    { { 2.0, 2.0 }, { 2.0, 2.0 } },
    { { 6.0, 2.0 }, { 2.0, 6.0 } }
};

// P[état, action du rang, action de la colonne, état suivant].
// Dans la version initiale, seule l'action du rang choisit la transition.
double[,,,] Transitions = new double[2, 2, 2, 2];
for (int b = 0; b < 2; b++)
{
    Transitions[0, 0, b, 0] = 1.0;
    Transitions[0, 1, b, 1] = 1.0;
    Transitions[1, 0, b, 1] = 1.0;
    Transitions[1, 1, b, 1] = 1.0;
}
Console.WriteLine($"États : {Gains.GetLength(0)}; actions du rang : {Gains.GetLength(1)}; actions de la colonne : {Gains.GetLength(2)}.");
for (int s = 0; s < Gains.GetLength(0); s++)
    Console.WriteLine($"G[{s}] = [[{Gains[s,0,0]:0.##}, {Gains[s,0,1]:0.##}], [{Gains[s,1,0]:0.##}, {Gains[s,1,1]:0.##}]]");

Runtime .NET : 10.0.15; noyau : .NET Interactive / C#; framework cible du dépôt : .NET 9+


États : 2; actions du rang : 2; actions de la colonne : 2.


G[0] = [[2, 2], [2, 2]]


G[1] = [[6, 2], [2, 6]]


Le modèle comporte deux états et deux actions par joueur. À l’état 0, le rang peut rester ou sauter vers l’état 1; l’état 1 est absorbant. La colonne ne change pas encore la transition. Les gains sont identiques aux données du jumeau Python.

## 2. Valeur statique : résoudre le jeu d’une seule manche

La valeur statique $v_{\mathrm{stat}}(s)$ ignore les transitions futures et résout uniquement la matrice courante. C’est le calcul que fait un joueur myope, mais ce n’est pas encore ce qu’il obtient sur la partie : ses actions déplacent aussi l’état, et la section 4 l’évalue sur la partie entière. Pour une matrice 2×2, on vérifie d’abord s’il existe une selle pure : la garantie maximin du rang égale alors la garantie minimax de la colonne. La formule fermée des stratégies mixtes s’applique seulement lorsqu’aucune selle pure n’existe.

In [2]:
record StaticSolution(double Value, double[] RowMix, double[] ColumnMix, bool HasPureSaddle);

static StaticSolution SolveStatic2x2(double[,] matrix, double tolerance = 1e-9)
{
    if (!double.IsFinite(tolerance) || tolerance <= 0.0)
        throw new ArgumentOutOfRangeException(nameof(tolerance), "La tolérance doit être finie et strictement positive.");
    if (matrix.GetLength(0) != 2 || matrix.GetLength(1) != 2)
        throw new ArgumentException("La résolution analytique attend une matrice 2×2.");
    if (matrix.Cast<double>().Any(value => !double.IsFinite(value)))
        throw new ArgumentException("Les gains de la matrice doivent tous être finis.");
    double row0Minimum = Math.Min(matrix[0, 0], matrix[0, 1]);
    double row1Minimum = Math.Min(matrix[1, 0], matrix[1, 1]);
    double maximin = Math.Max(row0Minimum, row1Minimum);
    double col0Maximum = Math.Max(matrix[0, 0], matrix[1, 0]);
    double col1Maximum = Math.Max(matrix[0, 1], matrix[1, 1]);
    double minimax = Math.Min(col0Maximum, col1Maximum);

    if (Math.Abs(maximin - minimax) <= tolerance)
    {
        int row = row0Minimum >= row1Minimum ? 0 : 1;
        int col = col0Maximum <= col1Maximum ? 0 : 1;
        double[] pPure = { 0.0, 0.0 };
        double[] qPure = { 0.0, 0.0 };
        pPure[row] = 1.0;
        qPure[col] = 1.0;
        return new StaticSolution(maximin, pPure, qPure, true);
    }

    double a = matrix[0, 0], b = matrix[0, 1];
    double c = matrix[1, 0], d = matrix[1, 1];
    double denominator = a - b - c + d;
    if (!double.IsFinite(denominator) || Math.Abs(denominator) <= tolerance)
        throw new InvalidOperationException("Matrice 2×2 dégénérée : dénominateur non fini ou nul sans selle détectée.");
    double[] p = { (d - c) / denominator, (a - b) / denominator };
    double[] q = { (d - b) / denominator, (a - c) / denominator };
    if (p.Any(x => !double.IsFinite(x) || x < -tolerance || x > 1 + tolerance) ||
        q.Any(x => !double.IsFinite(x) || x < -tolerance || x > 1 + tolerance) ||
        Math.Abs(p.Sum() - 1.0) > tolerance || Math.Abs(q.Sum() - 1.0) > tolerance)
        throw new InvalidOperationException("Les probabilités calculées ne sont pas des distributions du simplexe.");
    double value = (a * d - b * c) / denominator;
    if (!double.IsFinite(value)) throw new InvalidOperationException("La valeur du jeu matriciel n'est pas finie.");
    return new StaticSolution(value, p, q, false);
}

// Témoin discriminant : cette matrice possède une selle pure en ligne 0 / colonne 1.
double[,] SaddleWitness = { { 4.0, 3.0 }, { 2.0, 0.0 } };
StaticSolution witnessSolution = SolveStatic2x2(SaddleWitness);
if (Math.Abs(witnessSolution.Value - 3.0) > 1e-9 || !witnessSolution.HasPureSaddle ||
    !witnessSolution.RowMix.SequenceEqual(new[] { 1.0, 0.0 }) ||
    !witnessSolution.ColumnMix.SequenceEqual(new[] { 0.0, 1.0 }))
    throw new InvalidOperationException("Le témoin de selle pure n'est pas résolu correctement.");
Console.WriteLine($"Témoin [[4,3],[2,0]] : valeur={witnessSolution.Value:0.0000}, p={string.Join('/', witnessSolution.RowMix)}, q={string.Join('/', witnessSolution.ColumnMix)}.");

Témoin [[4,3],[2,0]] : valeur=3.0000, p=1/0, q=0/1.


Le témoin [[4,3],[2,0]] distingue les deux branches du solveur : la valeur de selle est bien bornée par les gains et les probabilités sont pures. L’appliquer directement à la formule de mélange produirait des probabilités hors du simplexe; la détection de selle doit donc précéder le calcul mixte.

In [3]:
static double[,] MatrixAt(double[,,] tensor, int state)
{
    return new double[,] { { tensor[state, 0, 0], tensor[state, 0, 1] },
                           { tensor[state, 1, 0], tensor[state, 1, 1] } };
}
StaticSolution staticState0 = SolveStatic2x2(MatrixAt(Gains, 0));
StaticSolution staticState1 = SolveStatic2x2(MatrixAt(Gains, 1));
double[] StaticValues = { staticState0.Value, staticState1.Value };
Console.WriteLine($"Valeur statique état 0 : {staticState0.Value:0.0000}; selle pure={staticState0.HasPureSaddle}.");
Console.WriteLine($"Valeur statique état 1 : {staticState1.Value:0.0000}; selle pure={staticState1.HasPureSaddle}; p=({staticState1.RowMix[0]:0.000}, {staticState1.RowMix[1]:0.000}).");

Valeur statique état 0 : 2.0000; selle pure=True.


Valeur statique état 1 : 4.0000; selle pure=False; p=(0.500, 0.500).


La matrice constante de l’état 0 a une selle pure et une valeur de 2. La matrice de l’état 1 n’a pas de selle pure; son équilibre mixte donne la valeur statique 4.

### Exercice 1 — le jeu à un seul état se réduit au minimax

Écrivez une fonction qui prend un callback d’itération, une matrice 2×2 et un facteur $\gamma$. Elle doit construire le jeu à un état avec une transition de probabilité 1 vers lui-même, exécuter le callback, puis renvoyer la valeur obtenue et le nombre d’itérations.

**Indice :** comparez ensuite la valeur renvoyée à la valeur du jeu matriciel statique. La garde appelle votre fonction sur plusieurs matrices, dont une selle pure et un équilibre mixte, à plusieurs facteurs d’actualisation.

In [4]:
record StaticConsistencyAttempt(bool Completed, double StochasticValue, int Iterations);
static StaticConsistencyAttempt StaticConsistency(
    Func<double[,,], double[,,,], double, ValueIterationResult> iterate,
    double[,] matrix, double gamma)
{
    // TODO étudiant : construire G/P à un état et appeler le callback iterate.
    return new StaticConsistencyAttempt(false, double.NaN, 0);
}

double[,] ExercisePureMatrix = { { 4.0, 3.0 }, { 2.0, 0.0 } };
double[,] ExerciseMixedMatrix = { { 1.0, -1.0 }, { -1.0, 1.0 } };
double[,] ExerciseSecondSaddleMatrix = { { 5.0, 1.0 }, { 3.0, 2.0 } };
Console.WriteLine("Exercice 1 à compléter : jeu à un état, callback d’itération et comparaison statique.");

Exercice 1 à compléter : jeu à un état, callback d’itération et comparaison statique.


## 3. Itération de valeur

On calcule maintenant la **valeur dynamique** $V$, celle du jeu stochastique entier, transitions comprises. À chaque tour, le gain courant est normalisé par $(1-\gamma)$ et la continuation actualisée par $\gamma$. Le calcul de la valeur de chaque matrice de continuation reste un vrai minimax mixte. On arrête lorsque la norme infinie entre deux vecteurs successifs passe sous la tolérance, ou lorsqu’on atteint le plafond d’itérations.

In [5]:
static double[] BellmanOperator(double[] values, double[,,] gains, double[,,,] transitions, double gamma)
{
    int states = gains.GetLength(0), rows = gains.GetLength(1), columns = gains.GetLength(2);
    if (values.Length != states || values.Any(x => !double.IsFinite(x)))
        throw new ArgumentException("Le vecteur V doit contenir une valeur finie par état.");
    double[] updated = new double[states];
    for (int s = 0; s < states; s++)
    {
        var continuationGame = new double[rows, columns];
        for (int a = 0; a < rows; a++)
        for (int b = 0; b < columns; b++)
        {
            double future = 0.0;
            for (int next = 0; next < states; next++)
                future += transitions[s, a, b, next] * values[next];
            continuationGame[a, b] = (1.0 - gamma) * gains[s, a, b] + gamma * future;
            if (!double.IsFinite(continuationGame[a, b]))
                throw new InvalidOperationException("Le gain de continuation n'est pas fini.");
        }
        updated[s] = SolveStatic2x2(continuationGame).Value;
        if (!double.IsFinite(updated[s])) throw new InvalidOperationException("La valeur de Bellman n'est pas finie.");
    }
    return updated;
}

static ValueIterationResult ValueIteration(double[,,] gains, double[,,,] transitions, double gamma,
    double tolerance = 1e-9, int maxIterations = 20000, double[] initial = null)
{
    if (!double.IsFinite(gamma) || gamma < 0.0 || gamma >= 1.0 ||
        !double.IsFinite(tolerance) || tolerance <= 0.0 || maxIterations < 1)
        throw new ArgumentOutOfRangeException(nameof(gamma), "Il faut un gamma fini dans [0,1), une tolérance finie positive et un plafond positif.");
    int states = gains.GetLength(0);
    if (states < 1 || gains.GetLength(1) != 2 || gains.GetLength(2) != 2 ||
        transitions.GetLength(0) != states || transitions.GetLength(1) != gains.GetLength(1) ||
        transitions.GetLength(2) != gains.GetLength(2) || transitions.GetLength(3) != states)
        throw new ArgumentException("G doit avoir au moins un état et deux actions par joueur; les dimensions de P doivent correspondre.");
    if (gains.Cast<double>().Any(value => !double.IsFinite(value)))
        throw new ArgumentException("Tous les gains G doivent être finis.");
    for (int s = 0; s < states; s++)
    for (int a = 0; a < gains.GetLength(1); a++)
    for (int b = 0; b < gains.GetLength(2); b++)
    {
        double mass = 0.0;
        for (int next = 0; next < states; next++)
        {
            double probability = transitions[s, a, b, next];
            if (!double.IsFinite(probability) || probability < 0.0)
                throw new ArgumentException($"La transition ({s},{a},{b},{next}) doit être finie et non négative.");
            mass += probability;
        }
        if (!double.IsFinite(mass) || Math.Abs(mass - 1.0) > 1e-9)
            throw new ArgumentException($"La transition ({s},{a},{b}) a une masse {mass:0.########}, attendue 1.");
    }
    double[] value = initial is null ? new double[states] : (double[])initial.Clone();
    if (value.Length != states || value.Any(x => !double.IsFinite(x)))
        throw new ArgumentException("Le vecteur initial doit avoir une valeur finie par état.");
    var history = new List<double[]> { (double[])value.Clone() };
    double residual = double.PositiveInfinity;
    for (int iteration = 1; iteration <= maxIterations; iteration++)
    {
        double[] nextValue = BellmanOperator(value, gains, transitions, gamma);
        residual = nextValue.Zip(value, (x, y) => Math.Abs(x - y)).Max();
        history.Add((double[])nextValue.Clone());
        value = nextValue;
        if (residual < tolerance)
            return new ValueIterationResult(value, iteration, history, true, residual);
    }
    return new ValueIterationResult(value, maxIterations, history, false, residual);
}

const double Gamma = 0.9, Tolerance = 1e-9;
const int MaxIterations = 20000;
ValueIterationResult baseline = ValueIteration(Gains, Transitions, Gamma, Tolerance, MaxIterations);
Console.WriteLine($"gamma={Gamma:0.00}; valeurs=[{string.Join(", ", baseline.Values.Select(x => x.ToString("0.0000", CultureInfo.InvariantCulture)))}].");
Console.WriteLine($"Arrêt numérique : {baseline.Iterations} itérations; résidu={baseline.Residual:E3}; convergé sous la tolérance={baseline.Converged}.");
Console.WriteLine("Repère analytique du modèle : les états absorbants à gain constant ont cette valeur constante; ici l'état 0 peut atteindre l'état 1.");

gamma=0.90; valeurs=[3.8000, 4.0000].


Arrêt numérique : 189 itérations; résidu=9.992E-010; convergé sous la tolérance=True.


Repère analytique du modèle : les états absorbants à gain constant ont cette valeur constante; ici l'état 0 peut atteindre l'état 1.


L’itération garde l’historique complet à partir de $V_0=0$. Le résidu mesuré sert de critère d’arrêt numérique. Le théorème de Shapley donne une garantie mathématique de point fixe unique et de convergence; le résultat calculé ici reste une approximation flottante arrêtée à la tolérance choisie, et ne constitue pas une preuve exacte.

**Contrôle indépendant de l’exercice 1.** La cellule suivante appelle le stub sur plusieurs matrices et facteurs, puis compare les valeurs renvoyées au solveur statique déjà exécuté.

In [6]:
static (double[,,] Gains, double[,,,] Transitions) MakeOneStateGame(double[,] matrix)
{
    var gains = new double[1, 2, 2];
    var transitions = new double[1, 2, 2, 1];
    for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++)
    {
        gains[0, a, b] = matrix[a, b];
        transitions[0, a, b, 0] = 1.0;
    }
    return (gains, transitions);
}
static bool SameTensor(Array actual, Array expected)
{
    if (actual is null || actual.Rank != expected.Rank) return false;
    for (int dimension = 0; dimension < actual.Rank; dimension++)
        if (actual.GetLength(dimension) != expected.GetLength(dimension)) return false;
    return actual.Cast<double>().SequenceEqual(expected.Cast<double>());
}
static bool AcceptStaticConsistency(StaticConsistencyAttempt result, double[,] matrix)
{
    if (!result.Completed || result.Iterations < 1 || result.Iterations > MaxIterations ||
        !double.IsFinite(result.StochasticValue)) return false;
    return Math.Abs(result.StochasticValue - SolveStatic2x2(matrix).Value) <= 1e-7;
}
static bool CheckStaticStudent(
    Func<Func<double[,,], double[,,,], double, ValueIterationResult>, double[,], double, StaticConsistencyAttempt> candidate,
    (double[,] Matrix, double Gamma)[] cases)
{
    int completed = 0;
    foreach (var test in cases)
    {
        var expectedGame = MakeOneStateGame(test.Matrix);
        int calls = 0, observedIterations = 0;
        double observedValue = double.NaN;
        ValueIterationResult MeasuredIteration(double[,,] g, double[,,,] p, double gamma)
        {
            if (!SameTensor(g, expectedGame.Gains) || !SameTensor(p, expectedGame.Transitions) || gamma != test.Gamma)
                throw new InvalidOperationException("Exercice 1 : le callback doit recevoir le jeu un-état et le gamma de la fixture.");
            var actual = ValueIteration(g, p, gamma, Tolerance, MaxIterations);
            calls++;
            observedValue = actual.Values[0];
            observedIterations = actual.Iterations;
            if (!actual.Converged) throw new InvalidOperationException("Exercice 1 : le callback n'a pas atteint sa tolérance.");
            return actual;
        }
        var result = candidate(MeasuredIteration, (double[,])test.Matrix.Clone(), test.Gamma);
        if (!result.Completed) continue;
        completed++;
        if (calls < 1 || !AcceptStaticConsistency(result, test.Matrix) ||
            Math.Abs(result.StochasticValue - observedValue) > 1e-12 || result.Iterations != observedIterations)
            throw new InvalidOperationException("Exercice 1 : résultat non fini, incorrect ou différent de l'itération réellement appelée.");
    }
    if (completed != 0 && completed != cases.Length)
        throw new InvalidOperationException("Exercice 1 : une réponse complète doit traiter toutes les fixtures.");
    return completed == cases.Length;
}
var oneStateCases = new[]
{
    (ExercisePureMatrix, 0.25), (ExercisePureMatrix, 0.85),
    (ExerciseMixedMatrix, 0.40), (ExerciseSecondSaddleMatrix, 0.70),
    (new double[,] {{ -2.0, -4.0 }, { -3.0, -1.0 }}, 0.60)
};
bool staticCompleted = CheckStaticStudent(StaticConsistency, oneStateCases);
Console.WriteLine(staticCompleted
    ? $"Exercice 1 contrôlé : {oneStateCases.Length} jeux/gammas, appels réels, valeurs et nombres d'itérations concordants."
    : $"Exercice 1 à compléter : garde prête pour {oneStateCases.Length} jeux/gammas ; aucun résultat étudiant certifié.");


Exercice 1 à compléter : garde prête pour 5 jeux/gammas ; aucun résultat étudiant certifié.


### Exercice 2 — étendre la valeur à trois états

Complétez l’itération de valeur pour trois états. La garde fournit plusieurs jeux à gains constants par état et transitions déterministes vers un état égal ou supérieur; les gains et les facteurs changent d’une fixture à l’autre.

**Indice :** la garde compare votre callback à une récurrence analytique indépendante propre à ces chaînes acycliques, puis recalcule elle-même le résidu de Bellman. Le nombre d’itérations déclaré n’est pas pris comme preuve de convergence.

Retournez aussi `Trace`, les vecteurs successifs depuis zéro, jusqu’au premier arrêt sous la tolérance. `Iterations` doit compter les transitions de cette trace. Le harnais recalcule chaque étape sur les fixtures, puis contrôle la valeur et le résidu ; un compteur déclaré sans trajectoire ne valide pas la réponse.


In [7]:
record ThreeStateAttempt(bool Completed, double[] Values, int Iterations, double[][] Trace);
record ThreeStateFixture(double[] Rewards, int[] Next, double Gamma, double[,,] Gains, double[,,,] Transitions);
static ThreeStateAttempt ValueIterationThreeStates(double[,,] gains3, double[,,,] transitions3, double gamma)
{
    // TODO étudiant : itérer jusqu'à la tolérance sur les trois états et retourner les valeurs.
    return new ThreeStateAttempt(false, Array.Empty<double>(), 0, Array.Empty<double[]>());
}
static ThreeStateFixture BuildThreeStateFixture(double[] rewards, int[] nextStates, double gamma)
{
    var gains = new double[3, 2, 2];
    var transitions = new double[3, 2, 2, 3];
    for (int s = 0; s < 3; s++) for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++)
    {
        gains[s, a, b] = rewards[s];
        transitions[s, a, b, nextStates[s]] = 1.0;
    }
    return new ThreeStateFixture(rewards, nextStates, gamma, gains, transitions);
}
static double[] OracleThreeState(ThreeStateFixture fixture)
{
    var expected = new double[3];
    for (int s = 2; s >= 0; s--)
    {
        int next = fixture.Next[s];
        if (next < s || next >= 3) throw new ArgumentException("La fixture oracle doit être acyclique ou absorbante.");
        expected[s] = next == s ? fixture.Rewards[s] :
            (1.0 - fixture.Gamma) * fixture.Rewards[s] + fixture.Gamma * expected[next];
    }
    return expected;
}
static double IndependentThreeStateResidual(ThreeStateFixture fixture, double[] values)
{
    double residual = 0.0;
    for (int s = 0; s < 3; s++)
    {
        double referenceTarget = double.NaN;
        for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++)
        {
            double future = 0.0;
            for (int next = 0; next < 3; next++) future += fixture.Transitions[s, a, b, next] * values[next];
            double target = (1.0 - fixture.Gamma) * fixture.Gains[s, a, b] + fixture.Gamma * future;
            if (!double.IsFinite(target)) return double.PositiveInfinity;
            if (double.IsNaN(referenceTarget)) referenceTarget = target;
            else if (Math.Abs(referenceTarget - target) > 1e-12) return double.PositiveInfinity;
        }
        residual = Math.Max(residual, Math.Abs(referenceTarget - values[s]));
    }
    return residual;
}
static bool AcceptThreeState(ThreeStateAttempt result, ThreeStateFixture fixture)
{
    if (!result.Completed || result.Values is null || result.Trace is null || result.Values.Length != 3 ||
        result.Iterations < 1 || result.Iterations > MaxIterations || result.Trace.Length != result.Iterations + 1 ||
        result.Values.Any(value => !double.IsFinite(value)) ||
        result.Trace.Any(step => step is null || step.Length != 3 || step.Any(value => !double.IsFinite(value)))) return false;
    if (result.Trace[0].Any(value => value != 0.0) ||
        result.Values.Zip(result.Trace[^1], (actual, last) => Math.Abs(actual - last)).Any(error => error > 1e-12)) return false;
    double lastDelta = double.PositiveInfinity;
    for (int k = 1; k < result.Trace.Length; k++)
    {
        var previous = result.Trace[k - 1];
        var current = result.Trace[k];
        for (int s = 0; s < 3; s++)
        {
            double expectedStep = (1.0 - fixture.Gamma) * fixture.Rewards[s] + fixture.Gamma * previous[fixture.Next[s]];
            if (Math.Abs(current[s] - expectedStep) > 1e-12) return false;
        }
        lastDelta = current.Zip(previous, (x, y) => Math.Abs(x - y)).Max();
        if (lastDelta < Tolerance && k < result.Trace.Length - 1) return false;
    }
    double[] expected = OracleThreeState(fixture);
    double maxError = result.Values.Zip(expected, (actual, target) => Math.Abs(actual - target)).Max();
    double actualResidual = IndependentThreeStateResidual(fixture, result.Values);
    return lastDelta < Tolerance && double.IsFinite(maxError) && maxError <= 1e-7 &&
        double.IsFinite(actualResidual) && actualResidual <= 1e-7;
}

ThreeStateFixture[] threeStateCases =
{
    BuildThreeStateFixture(new[] { 1.0, 3.0, 5.0 }, new[] { 1, 2, 2 }, 0.80),
    BuildThreeStateFixture(new[] { 4.0, 2.0, 6.0 }, new[] { 2, 2, 2 }, 0.55),
    BuildThreeStateFixture(new[] { 2.0, 1.5, 8.0 }, new[] { 0, 2, 2 }, 0.60),
    BuildThreeStateFixture(new[] { 0.5, 0.75, 1.2 }, new[] { 1, 2, 2 }, 0.92)
};
Func<double[,,], double[,,,], double, ThreeStateAttempt> studentThreeState = ValueIterationThreeStates;
ThreeStateAttempt[] threeStateResults = threeStateCases.Select(test => studentThreeState(test.Gains, test.Transitions, test.Gamma)).ToArray();
if (threeStateResults.All(result => !result.Completed))
    Console.WriteLine("Garde Exercice 2 : 4 jeux/G/P/gamma distincts ; trace, arrêt, oracle et résidu à vérifier ; stub en attente.");
else if (!threeStateCases.Zip(threeStateResults, (fixture, result) => AcceptThreeState(result, fixture)).All(ok => ok))
    throw new InvalidOperationException("Exercice 2 : une fixture trois états échoue à l'oracle ou au résidu indépendant.");
else Console.WriteLine($"Exercice 2 contrôlé : {threeStateResults.Length} réponses respectent leurs traces, critères d’arrêt et oracles indépendants.");

Garde Exercice 2 : 4 jeux/G/P/gamma distincts ; trace, arrêt, oracle et résidu à vérifier ; stub en attente.


## 4. Écart entre valeur statique et valeur dynamique

L’écart $V(s)-v_{\mathrm{stat}}(s)$ est la **différence entre valeur statique et valeur dynamique** : il compare la valeur d’une manche isolée à celle de la partie entière, transitions comprises. Il ne compare pas deux politiques jouées sur la même partie, et il ne mesure pas une **valeur de l’information** : les joueurs connaissent déjà l’état courant, l’information sur l’état est la même dans les deux calculs, et aucun signal n’est ajouté. À l’état 0, la partie peut conduire vers l’état haut; à l’état 1, l’état reste absorbant dans le futur, mais le flux normalisé constant a pour valeur 4, donc l’écart à la valeur statique est nul.

In [8]:
double[] StaticDynamicGap = baseline.Values.Zip(StaticValues, (v, stat) => v - stat).Select(gap => Math.Abs(gap) < 1e-12 ? 0.0 : gap).ToArray();
Console.WriteLine("État | valeur statique | valeur dynamique | écart");
for (int s = 0; s < StaticDynamicGap.Length; s++)
    Console.WriteLine($"{s,4} | {StaticValues[s],15:0.0000} | {baseline.Values[s],16:0.0000} | {StaticDynamicGap[s],8:0.0000}");
Console.WriteLine($"À gamma={Gamma:0.00}, l'écart dynamique − statique de l'état 0 suit 2γ={2.0 * Gamma:0.0000}; celui de l'état absorbant est {StaticDynamicGap[1]:0.0000}.");

record SvgPlot(string Markup);
Formatter.Register(typeof(SvgPlot), (obj, writer) => ((System.IO.TextWriter)writer).Write(((SvgPlot)obj).Markup), "text/html");
static string BuildShapleySvg(List<double[]> history, double[] gaps, double gamma)
{
    const int width = 1040, height = 440, plotTop = 78, plotBottom = 352, plotHeight = plotBottom - plotTop;
    var inv = CultureInfo.InvariantCulture;
    string F(double x) => x.ToString("0.##", inv);
    string Esc(string text) => SecurityElement.Escape(text) ?? "";
    var sb = new StringBuilder();
    sb.Append($"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}' font-family='sans-serif' font-size='12'>");
    sb.Append($"<rect width='{width}' height='{height}' fill='white'/>");
    void Text(double x, double y, string value, string extra = "") => sb.Append($"<text x='{F(x)}' y='{F(y)}' fill='#333333' {extra}>{Esc(value)}</text>");
    void Line(double x1, double y1, double x2, double y2, string color, string extra = "") => sb.Append($"<line x1='{F(x1)}' y1='{F(y1)}' x2='{F(x2)}' y2='{F(y2)}' stroke='{color}' {extra}/>");
    Text(260, 28, $"Convergence de l’itération de valeur (gamma={F(gamma)})", "text-anchor='middle' font-size='14' font-weight='bold'");
    Text(780, 28, "Écart entre valeur dynamique et valeur statique", "text-anchor='middle' font-size='14' font-weight='bold'");
    const double leftX = 68, leftWidth = 384, rightX = 604, rightWidth = 368;
    for (int tick = 0; tick <= 4; tick++)
    {
        double y = plotBottom - (tick / 4.5) * plotHeight;
        Line(leftX, y, leftX + leftWidth, y, "#E5E5E5");
        Text(leftX - 9, y + 4, tick.ToString(inv), "text-anchor='end'");
    }
    Line(leftX, plotTop, leftX, plotBottom, "#888888"); Line(leftX, plotBottom, leftX + leftWidth, plotBottom, "#888888");
    Text(leftX + leftWidth / 2, 408, "Itération", "text-anchor='middle'");
    Text(20, 218, "Valeur V(s)", "text-anchor='middle' transform='rotate(-90 20 218)'");
    int last = Math.Max(1, history.Count - 1);
    foreach (int i in new[] { 0, last / 2, last })
    {
        double x = leftX + leftWidth * i / last;
        Text(x, plotBottom + 18, i.ToString(inv), "text-anchor='middle'");
    }
    string[] colors = { "#4C72B0", "#55A868" };
    for (int state = 0; state < 2; state++)
    {
        string points = string.Join(" ", history.Select((v, i) => $"{F(leftX + leftWidth * i / last)},{F(plotBottom - Math.Clamp(v[state], 0.0, 4.5) / 4.5 * plotHeight)}"));
        sb.Append($"<polyline points='{points}' fill='none' stroke='{colors[state]}' stroke-width='2'/>");
        Line(leftX + 18 + state * 130, 54, leftX + 40 + state * 130, 54, colors[state], "stroke-width='2'");
        Text(leftX + 46 + state * 130, 58, $"V[{state}]");
    }
    for (int tick = 0; tick <= 4; tick++)
    {
        double value = tick * 0.5, y = plotBottom - value / 2.0 * plotHeight;
        Line(rightX, y, rightX + rightWidth, y, "#E5E5E5");
        Text(rightX - 9, y + 4, value.ToString("0.0", inv), "text-anchor='end'");
    }
    Line(rightX, plotTop, rightX, plotBottom, "#888888"); Line(rightX, plotBottom, rightX + rightWidth, plotBottom, "#888888");
    Text(rightX + rightWidth / 2, 408, "État", "text-anchor='middle'");
    Text(548, 218, "V(s) − v_stat(s)", "text-anchor='middle' transform='rotate(-90 548 218)'");
    for (int s = 0; s < gaps.Length; s++)
    {
        double center = rightX + rightWidth * (s + 0.5) / gaps.Length;
        double barHeight = Math.Clamp(gaps[s], 0.0, 2.0) / 2.0 * plotHeight;
        sb.Append($"<rect x='{F(center - 42)}' y='{F(plotBottom - barHeight)}' width='84' height='{F(barHeight)}' fill='{(s == 0 ? "#4C72B0" : "#999999")}'/>");
        Text(center, plotBottom + 18, $"État {s}", "text-anchor='middle'");
    }
    sb.Append("</svg>");
    return sb.ToString();
}
display(new SvgPlot(BuildShapleySvg(baseline.History, StaticDynamicGap, Gamma)));

État | valeur statique | valeur dynamique | écart


   0 |          2.0000 |           3.8000 |   1.8000


   1 |          4.0000 |           4.0000 |  -0.0000


À gamma=0.90, l'écart dynamique − statique de l'état 0 suit 2γ=1.8000; celui de l'état absorbant est -0.0000.


Convergence de l’itération de valeur (gamma=0.9) Écart entre valeur dynamique et valeur statique 0 1 2 3 4 Itération Valeur V(s) 0 94 189 <polyline points='68,352 70.03,339.82 72.06,317.9 74.1,298.17 76.13,280.42 78.16,264.44 80.19,250.06 82.22,237.11 84.25,225.46 86.29,214.98 88.32,205.54 90.35,197.05 92.38,189.41 94.41,182.53 96.44,176.34 98.48,170.77 100.51,165.75 102.54,161.24 104.57,157.18 106.6,153.52 108.63,150.23 110.67,147.27 112.7,144.61 114.73,142.21 116.76,140.05 118.79,138.11 120.83,136.36 122.86,134.78 124.89,133.37 126.92,132.09 128.95,130.95 130.98,129.91 133.02,128.99 135.05,128.15 137.08,127.4 139.11,126.72 141.14,126.11 143.17,125.56 145.21,125.07 147.24,124.62 149.27,124.22 151.3,123.86 153.33,123.54 155.37,123.25 157.4,122.98 159.43,122.75 161.46,122.54 163.49,122.34 165.52,122.17 167.56,122.02 169.59,121.88 171.62,121.75 173.65,121.64 175.68,121.54 177.71,121.45 179.75,121.36 181.78,121.29 183.81,121.22 185.84,121.16 187.87,121.11 189.9,121.06 191.94,121.02 193.97,120.98 196,120.94 198.03,120.91 200.06,120.88 202.1,120.85 204.13,120.83 206.16,120.81 208.19,120.79 210.22,120.77 212.25,120.76 214.29,120.75 216.32,120.73 218.35,120.72 220.38,120.71 222.41,120.7 224.44,120.7 226.48,120.69 228.51,120.68 230.54,120.68 232.57,120.67 234.6,120.67 236.63,120.66 238.67,120.66 240.7,120.65 242.73,120.65 244.76,120.65 246.79,120.65 248.83,120.64 250.86,120.64 252.89,120.64 254.92,120.64 256.95,120.64 258.98,120.63 261.02,120.63 263.05,120.63 265.08,120.63 267.11,120.63 269.14,120.63 271.17,120.63 273.21,120.63 275.24,120.63 277.27,120.63 279.3,120.63 281.33,120.63 283.37,120.63 285.4,120.63 287.43,120.63 289.46,120.62 291.49,120.62 293.52,120.62 295.56,120.62 297.59,120.62 299.62,120.62 301.65,120.62 303.68,120.62 305.71,120.62 307.75,120.62 309.78,120.62 311.81,120.62 313.84,120.62 315.87,120.62 317.9,120.62 319.94,120.62 321.97,120.62 324,120.62 326.03,120.62 328.06,120.62 330.1,120.62 332.13,120.62 334.16,120.62 336.19,120.62 338.22,120.62 340.25,120.62 342.29,120.62 344.32,120.62 346.35,120.62 348.38,120.62 350.41,120.62 352.44,120.62 354.48,120.62 356.51,120.62 358.54,120.62 360.57,120.62 362.6,120.62 364.63,120.62 366.67,120.62 368.7,120.62 370.73,120.62 372.76,120.62 374.79,120.62 376.83,120.62 378.86,120.62 380.89,120.62 382.92,120.62 384.95,120.62 386.98,120.62 389.02,120.62 391.05,120.62 393.08,120.62 395.11,120.62 397.14,120.62 399.17,120.62 401.21,120.62 403.24,120.62 405.27,120.62 407.3,120.62 409.33,120.62 411.37,120.62 413.4,120.62 415.43,120.62 417.46,120.62 419.49,120.62 421.52,120.62 423.56,120.62 425.59,120.62 427.62,120.62 429.65,120.62 431.68,120.62 433.71,120.62 435.75,120.62 437.78,120.62 439.81,120.62 441.84,120.62 443.87,120.62 445.9,120.62 447.94,120.62 449.97,120.62 452,120.62' fill='none' stroke='#4C72B0' stroke-width='2'/> V[0] <polyline points='68,352 70.03,327.64 72.06,305.72 74.1,286 76.13,268.24 78.16,252.26 80.19,237.88 82.22,224.94 84.25,213.29 86.29,202.8 88.32,193.37 90.35,184.87 92.38,177.23 94.41,170.35 96.44,164.16 98.48,158.59 100.51,153.58 102.54,149.06 104.57,145 106.6,141.35 108.63,138.06 110.67,135.09 112.7,132.43 114.73,130.03 116.76,127.87 118.79,125.93 120.83,124.18 122.86,122.61 124.89,121.19 126.92,119.92 128.95,118.77 130.98,117.74 133.02,116.81 135.05,115.97 137.08,115.22 139.11,114.54 141.14,113.93 143.17,113.38 145.21,112.89 147.24,112.44 149.27,112.04 151.3,111.68 153.33,111.36 155.37,111.07 157.4,110.81 159.43,110.57 161.46,110.36 163.49,110.17 165.52,109.99 167.56,109.84 169.59,109.7 171.62,109.57 173.65,109.46 175.68,109.36 177.71,109.27 179.75,109.19 181.78,109.11 183.81,109.04 185.84,108.98 187.87,108.93 189.9,108.88 191.94,108.84 193.97,108.8 196,108.76 198.03,108.73 200.06,108.7 202.1,108.68 204.13,108.65 206.16,108.63 208.19,108.61 210.22,108.6 212.25,108.58 214.29,108.57 216.32,108.56 218.35,108.54 220.38,108.53 222.41,108.53 224.44,108.52 226.48,108.51 228.51,108.5 230.54,108.5 232.57,108.49 234.6,108.49 236.63,108.48 238.67,108.48 240.7,108.48 242.73,1

**Lecture de la figure.** Le panneau gauche montre la suite des estimations des deux états; la courbe de l’état absorbant rejoint sa valeur normalisée constante. Le panneau droit montre, par état, l’écart entre valeur dynamique et valeur statique. Dans l’état 0, la valeur dynamique dépasse la valeur statique parce que la partie peut conduire à l’état 1; dans l’état 1, la poursuite du même état compte bien dans la valeur, mais le jeu futur ne change pas la valeur par période de 4.

Cet écart dit combien la partie entière vaut de plus qu’une manche isolée. Il ne dit pas encore combien gagne un joueur qui planifie par rapport à un joueur myope : pour le savoir, il faut évaluer le joueur myope sur la même partie.

La méthode donne une série d’approximations et un résidu fini. L’égalité analytique du cas construit sert de repère; la tolérance et l’arithmétique flottante ne transforment pas le calcul en preuve exacte.

### Le joueur myope, évalué sur la même partie

Un joueur myope choisit, à chaque état, une action optimale pour la seule matrice courante. À l’état 0, cette matrice est constante : **rester** et **sauter** garantissent toutes deux 2, et sont donc toutes deux myopiquement optimales. La politique myope n’est définie qu’une fois ce **départage** fixé. À l’état 1, le joueur myope joue l’équilibre mixte de $G[1]$.

La cellule suivante évalue chaque politique myope sur la même partie que $V$ : même critère actualisé et normalisé, même $\gamma$. Le rang joue sa politique $\pi$ sans la réoptimiser, la colonne répond au mieux à chaque état :

$$W(s) = \min_{b} \sum_a \pi_s(a)\Big[(1-\gamma)\,G[s,a,b] + \gamma \sum_{s'} P[s,a,b,s']\,W(s')\Big]$$

L’écart $V(0)-W(0)$ est alors le **gain de planification** de cette politique myope : ce que perd ce joueur, sur la même partie, face à la valeur optimale.

In [9]:
static (double[] Values, int Iterations) EvaluateRowPolicy(double[][] policy, double[,,] gains,
    double[,,,] transitions, double gamma, double tolerance = 1e-9, int maxIterations = 20000)
{
    // Valeur, sur la partie entière, d'une politique stationnaire FIXE du rang : policy[s][a].
    // Même critère que V (gains normalisés par 1-gamma, avenir actualisé par gamma) ;
    // le rang ne réoptimise pas, la colonne répond au mieux à chaque état.
    int states = gains.GetLength(0), rows = gains.GetLength(1), columns = gains.GetLength(2);
    double[] w = new double[states];
    for (int iteration = 1; iteration <= maxIterations; iteration++)
    {
        double[] next = new double[states];
        for (int s = 0; s < states; s++)
        {
            double worst = double.PositiveInfinity;
            for (int b = 0; b < columns; b++)
            {
                double expected = 0.0;
                for (int a = 0; a < rows; a++)
                {
                    double future = 0.0;
                    for (int t = 0; t < states; t++) future += transitions[s, a, b, t] * w[t];
                    expected += policy[s][a] * ((1.0 - gamma) * gains[s, a, b] + gamma * future);
                }
                worst = Math.Min(worst, expected);
            }
            next[s] = worst;
        }
        double residual = next.Zip(w, (x, y) => Math.Abs(x - y)).Max();
        w = next;
        if (residual < tolerance) return (w, iteration);
    }
    return (w, maxIterations);
}

// À l'état 0, chaque action pure du rang garantit-elle la valeur statique de G[0] ?
double stayGuarantee = Math.Min(Gains[0, 0, 0], Gains[0, 0, 1]);
double jumpGuarantee = Math.Min(Gains[0, 1, 0], Gains[0, 1, 1]);
Console.WriteLine($"État 0 : garantie de « rester » = {stayGuarantee:0.0}, de « sauter » = {jumpGuarantee:0.0}, valeur statique = {StaticValues[0]:0.0} : les deux actions sont myopiquement optimales.");

// Deux politiques myopes : même jeu à l'état 1 (équilibre de G[1]), départage opposé à l'état 0.
var myopicPolicies = new (string Name, double[][] Policy)[]
{
    ("myope qui reste", new[] { new[] { 1.0, 0.0 }, staticState1.RowMix }),
    ("myope qui saute", new[] { new[] { 0.0, 1.0 }, staticState1.RowMix })
};
Console.WriteLine($"Valeur dynamique optimale V(0) = {baseline.Values[0]:0.0000} (valeur statique v_stat(0) = {StaticValues[0]:0.0000}).");
Console.WriteLine("Politique       | W(0) sur la partie | V(0) − W(0)");
var myopicPolicyValues = new Dictionary<string, double>();
foreach (var (name, policy) in myopicPolicies)
{
    var evaluation = EvaluateRowPolicy(policy, Gains, Transitions, Gamma, Tolerance, MaxIterations);
    double planningGain = baseline.Values[0] - evaluation.Values[0];
    if (Math.Abs(planningGain) < 1e-6) planningGain = 0.0;   // bruit flottant sous la tolérance d'arrêt
    myopicPolicyValues[name] = evaluation.Values[0];
    Console.WriteLine($"{name,-15} | {evaluation.Values[0],18:0.0000} | {planningGain,11:0.0000}");
}
double expectedJump = (1.0 - Gamma) * StaticValues[0] + Gamma * StaticValues[1];
Console.WriteLine($"Contrôle analytique du myope qui saute : (1−γ)·2 + γ·4 = {expectedJump:0.0000}.");
if (Math.Abs(myopicPolicyValues["myope qui reste"] - StaticValues[0]) > 1e-6 ||
    Math.Abs(myopicPolicyValues["myope qui saute"] - expectedJump) > 1e-6)
    throw new InvalidOperationException("Le témoin des deux départages myopes ne retrouve pas 2 et 3,8.");

État 0 : garantie de « rester » = 2.0, de « sauter » = 2.0, valeur statique = 2.0 : les deux actions sont myopiquement optimales.


Valeur dynamique optimale V(0) = 3.8000 (valeur statique v_stat(0) = 2.0000).


Politique       | W(0) sur la partie | V(0) − W(0)


myope qui reste |             2.0000 |      1.8000


myope qui saute |             3.8000 |      0.0000


Contrôle analytique du myope qui saute : (1−γ)·2 + γ·4 = 3.8000.


**Lecture du résultat.** Les deux départages myopes donnent deux réponses opposées. Le myope qui reste obtient 2 sur la partie, soit 1,8 de moins que la valeur optimale. Le myope qui saute obtient 0,1 × 2 + 0,9 × 4 = 3,8, exactement la valeur optimale : son gain de planification est nul.

L’écart de 1,8 entre valeur statique et valeur dynamique ne mesure donc pas, à lui seul, ce que rapporte la planification : il coïncide avec le gain de planification du seul myope qui reste. Dans ce modèle, planifier ne sert qu’à départager deux actions que la matrice courante ne distingue pas. Mesurer un gain de planification demande de fixer une politique myope complète, départage compris, et de l’évaluer sur le même horizon que la politique optimale. Mesurer une valeur de l’information demanderait davantage : un signal, ce qu’il révèle, et des politiques qui en dépendent. Rien de tel ici, où l’état est connu de tous.

### Exercice 3 — mesurer l’écart entre valeur dynamique et valeur statique selon $\gamma$

Complétez une fonction qui reçoit un callback d’itération, des tableaux de gains et de transitions, et une grille de facteurs. Elle doit calculer pour chaque facteur l’écart entre la valeur dynamique de l’état 0 et sa valeur statique.

**Indice :** la garde exécute votre fonction sur deux jeux différents et plusieurs facteurs. Elle compare chaque point à un oracle analytique calculé depuis les matrices statiques et les transitions, pas seulement à une pente ou à une forme croissante.

In [10]:
record StaticDynamicGapCurveAttempt(bool Completed, double[] Values);
static StaticDynamicGapCurveAttempt StaticDynamicGapCurve(
    Func<double[,,], double[,,,], double, ValueIterationResult> iterate,
    double[,,] gains, double[,,,] transitions, double[] gammas)
{
    // TODO étudiant : appeler iterate pour chaque gamma, puis calculer V(0)-v_stat(0).
    return new StaticDynamicGapCurveAttempt(false, Array.Empty<double>());
}
static double[] OracleStaticDynamicGapCurve(double[,,] gains, double[] gammas)
{
    // Oracle analytique pour les fixtures prescrites : l'état haut est absorbant et plus rémunérateur.
    double static0 = SolveStatic2x2(MatrixAt(gains, 0)).Value;
    double static1 = SolveStatic2x2(MatrixAt(gains, 1)).Value;
    return gammas.Select(gamma => gamma * (static1 - static0)).ToArray();
}
static bool AcceptStaticDynamicGapCurve(StaticDynamicGapCurveAttempt result, double[,,] gains, double[] gammas)
{
    if (!result.Completed || result.Values.Length != gammas.Length ||
        result.Values.Any(value => !double.IsFinite(value)) ||
        gammas.Any(gamma => !double.IsFinite(gamma) || gamma < 0.0 || gamma >= 1.0)) return false;
    double[] oracle = OracleStaticDynamicGapCurve(gains, gammas);
    // Borne issue de la contraction : la tolérance sur deux itérés n'est pas l'erreur sur la valeur.
    return Enumerable.Range(0, gammas.Length).All(i =>
        Math.Abs(result.Values[i] - oracle[i]) <= Tolerance / (1.0 - gammas[i]) + 1e-9);
}
static bool CheckStaticDynamicGapStudent(
    Func<Func<double[,,], double[,,,], double, ValueIterationResult>, double[,,], double[,,,], double[], StaticDynamicGapCurveAttempt> candidate,
    (double[,,] Gains, double[,,,] Transitions, double[] Gammas)[] cases)
{
    int completed = 0;
    foreach (var test in cases)
    {
        var expectedGains = (double[,,])test.Gains.Clone();
        var expectedTransitions = (double[,,,])test.Transitions.Clone();
        var expectedGammas = (double[])test.Gammas.Clone();
        int[] calls = new int[expectedGammas.Length];
        double[] observedGaps = Enumerable.Repeat(double.NaN, expectedGammas.Length).ToArray();
        double staticValue0 = SolveStatic2x2(MatrixAt(expectedGains, 0)).Value;
        ValueIterationResult MeasuredIteration(double[,,] g, double[,,,] p, double gamma)
        {
            int index = Array.IndexOf(expectedGammas, gamma);
            if (index < 0 || !SameTensor(g, expectedGains) || !SameTensor(p, expectedTransitions))
                throw new InvalidOperationException("Exercice 3 : le callback reçoit un jeu ou un gamma différent de la fixture.");
            var actual = ValueIteration(g, p, gamma, Tolerance, MaxIterations);
            if (!actual.Converged) throw new InvalidOperationException("Exercice 3 : le callback n'a pas atteint sa tolérance.");
            calls[index]++;
            observedGaps[index] = actual.Values[0] - staticValue0;
            return actual;
        }
        var result = candidate(MeasuredIteration, (double[,,])test.Gains.Clone(),
            (double[,,,])test.Transitions.Clone(), (double[])test.Gammas.Clone());
        if (!result.Completed) continue;
        completed++;
        if (calls.Any(count => count < 1) || !AcceptStaticDynamicGapCurve(result, expectedGains, expectedGammas) ||
            result.Values.Zip(observedGaps, (actual, observed) => Math.Abs(actual - observed)).Any(error => !double.IsFinite(error) || error > 1e-12))
            throw new InvalidOperationException("Exercice 3 : chaque point doit provenir de l'itération réellement appelée et correspondre à l'oracle.");
    }
    if (completed != 0 && completed != cases.Length)
        throw new InvalidOperationException("Exercice 3 : une réponse complète doit traiter les deux jeux.");
    return completed == cases.Length;
}
double[,,] AlternateGains = new double[2, 2, 2]
{
    { { 1.0, 1.0 }, { 1.0, 1.0 } },
    { { 7.0, 3.0 }, { 3.0, 7.0 } }
};
double[] baseGammas = { 0.50, 0.65, 0.80, 0.90, 0.99 };
double[] alternateGammas = { 0.35, 0.60, 0.85 };
var staticDynamicGapCases = new[] { (Gains, Transitions, baseGammas), (AlternateGains, Transitions, alternateGammas) };
bool curveCompleted = CheckStaticDynamicGapStudent(StaticDynamicGapCurve, staticDynamicGapCases);
Console.WriteLine(curveCompleted
    ? "Exercice 3 contrôlé : deux jeux, chaque gamma appelé et chaque point comparé au résultat réel et à l'oracle."
    : "Exercice 3 à compléter : garde prête pour deux jeux et huit facteurs gamma ; aucun résultat étudiant certifié.");


Exercice 3 à compléter : garde prête pour deux jeux et huit facteurs gamma ; aucun résultat étudiant certifié.


## 5. Quand la colonne conteste la transition

Modifions les transitions : depuis l’état 0, l’action de saut du rang mène à l’état 1 seulement si la colonne choisit de ne pas bloquer. L’opérateur doit alors résoudre les actions conjointement dans la matrice de continuation; la colonne agit à la fois sur le paiement du tour et sur l’état futur.

In [11]:
double[,,,] ContestedTransitions = new double[2, 2, 2, 2];
for (int a = 0; a < 2; a++)
{
    ContestedTransitions[1, a, 0, 1] = 1.0;
    ContestedTransitions[1, a, 1, 1] = 1.0;
    ContestedTransitions[0, a, 0, a == 1 ? 1 : 0] = 1.0;
    ContestedTransitions[0, a, 1, 0] = 1.0;
}
ValueIterationResult contested = ValueIteration(Gains, ContestedTransitions, Gamma, Tolerance, MaxIterations);
Console.WriteLine($"Sans contestation : V(0)={baseline.Values[0]:0.0000}; transition contestée : V(0)={contested.Values[0]:0.0000}.");
Console.WriteLine($"Valeurs contestées=[{string.Join(", ", contested.Values.Select(x => x.ToString("0.0000", CultureInfo.InvariantCulture)))}]; {contested.Iterations} itérations, résidu={contested.Residual:E3}.");

Sans contestation : V(0)=3.8000; transition contestée : V(0)=2.0000.


Valeurs contestées=[2.0000, 4.0000]; 189 itérations, résidu=9.992E-010.


Lorsque la colonne peut bloquer le saut, la valeur de l’état 0 retombe à la valeur statique de sa matrice : l’écart entre valeur dynamique et valeur statique disparaît entièrement, et aucun départage ne permet plus au rang d’atteindre l’état haut. L’avenir est ici contestable dans le jeu lui-même; il ne s’agit pas d’un paramètre indépendant du paiement. Les deux itérations sont numériques et leur comparaison dépend des mêmes paramètres et du même critère d’arrêt.

## 6. Lien avec l’apprentissage par renforcement

Dans ce petit jeu fini, l’itération de valeur calcule une référence sous le modèle connu. Un algorithme d’apprentissage par renforcement pourrait tenter d’estimer une valeur ou une politique à partir d’échantillons, mais le résultat de ce carnet ne promet pas la convergence d’un algorithme RL particulier : celle-ci dépend de ses hypothèses, de ses données et de sa règle d’apprentissage.

## À retenir

- La valeur d’un jeu stochastique actualisé combine le gain courant et les transitions futures dans un minimax de continuation.
- La selle pure doit être détectée avant d’appliquer les formules d’équilibre mixte d’une matrice 2×2.
- L’écart entre valeur statique et valeur dynamique compare une manche isolée à la partie entière. Ce n’est ni une valeur de l’information, ni à lui seul un gain de planification : évalué sur la même partie, le joueur myope obtient 2 s’il reste et 3,8 s’il saute, selon son départage.
- Un état absorbant continue à rapporter son flux futur; avec les gains normalisés de ce modèle, sa valeur reste 4.
- L’itération converge théoriquement dans le cadre posé, tandis que le carnet calcule un résultat flottant à tolérance finie. Aucun résultat ne garantit à lui seul la convergence d’un algorithme RL.

## Vérifiez votre compréhension

1. Pourquoi faut-il vérifier la selle pure avant la formule mixte ?
<details><summary>Réponse</summary>
La formule mixte à plein support n’est valide que lorsque la matrice n’a pas de point selle pur; appliquée au témoin avec selle, elle peut produire des probabilités hors du simplexe.
</details>

2. Pourquoi l’état absorbant garde-t-il une valeur de 4 même si son écart entre valeur dynamique et valeur statique est nul ?
<details><summary>Réponse</summary>
L’état continue d’être visité à chaque période. Le paiement normalisé constant vaut 4 au point fixe; l’écart nul signifie seulement que la valeur statique de sa matrice est déjà 4.
</details>

3. L’écart entre valeur statique et valeur dynamique mesure-t-il un gain de planification, ou une valeur de l’information ?
<details><summary>Réponse</summary>
Ni l’un ni l’autre. Un gain de planification se mesure en évaluant une politique myope sur la même partie que la politique optimale : il vaut 1,8 pour le myope qui reste et 0 pour le myope qui saute, selon le départage. Une valeur de l’information suppose un signal que reçoit un joueur; ici l’état est connu de tous.
</details>

## Pour aller plus loin

- Modifier les probabilités de transition et vérifier comment le contrôle de la colonne change la valeur.
- Étudier des jeux avec plus de deux actions; le solveur analytique 2×2 devra alors être remplacé par un algorithme de jeu matriciel adapté.
- Pour mesurer une valeur de l’information, construire un jeu où un signal révèle partiellement l’état, puis comparer des politiques qui en dépendent à des politiques qui l’ignorent.
- Comparer sur une simulation le calcul exact du petit modèle à des estimations apprises, en séparant les hypothèses de convergence propres à chaque méthode.

## Références

- Shapley, L. S. (1953). *Stochastic Games*. Proceedings of the National Academy of Sciences, 39(10), 1095–1100.
- Puterman, M. L. (2005). *Markov Decision Processes: Discrete Stochastic Dynamic Programming*. Wiley.
- Carnet **05** (minimax), **Dynamics-01** (jeux répétés) et **17** (multi-agent RL) de cette série.